In [1]:
from pyspark.sql import SparkSession
import spark

spark = SparkSession. builder \
.appName("Spark_Table_Cache") \
.getOrCreate()

Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/14 16:52:06 INFO SparkEnv: Registering MapOutputTracker
26/04/14 16:52:06 INFO SparkEnv: Registering BlockManagerMaster
26/04/14 16:52:06 INFO SparkEnv: Registering BlockManagerMasterHeartbeat
26/04/14 16:52:06 INFO SparkEnv: Registering OutputCommitCoordinator


In [2]:
customers_schema = 'customers_id INT, name STRING, city STRING, state STRING, country STRING, registration_date STRING, is_active BOOLEAN'

In [3]:
customers_df = spark.read \
.format('csv') \
.schema(customers_schema) \
.load('/data/customers_300mb.csv')

In [4]:
customers_df.write.format('csv').saveAsTable('default.customers_300mb')

ivysettings.xml file not found in HIVE_HOME or HIVE_CONF_DIR,/etc/hive/conf.dist/ivysettings.xml will be used
26/04/14 16:54:38 WARN HiveExternalCatalog: Couldn't find corresponding Hive SerDe for data source provider csv. Persisting data source table `spark_catalog`.`default`.`customers_300mb` into Hive metastore in Spark SQL specific format, which is NOT compatible with Hive.
26/04/14 16:54:38 WARN SessionState: METASTORE_FILTER_HOOK will be ignored, since hive.security.authorization.manager is set to instance of HiveAuthorizerFactory.


In [5]:
spark.sql('describe customers_300mb').show()

+-----------------+---------+-------+
|         col_name|data_type|comment|
+-----------------+---------+-------+
|     customers_id|      int|   NULL|
|             name|   string|   NULL|
|             city|   string|   NULL|
|            state|   string|   NULL|
|          country|   string|   NULL|
|registration_date|   string|   NULL|
|        is_active|  boolean|   NULL|
+-----------------+---------+-------+



In [6]:
spark.sql('describe extended customers_300mb').show(truncate =False)

+----------------------------+---------------------------------------------------------+-------+
|col_name                    |data_type                                                |comment|
+----------------------------+---------------------------------------------------------+-------+
|customers_id                |int                                                      |NULL   |
|name                        |string                                                   |NULL   |
|city                        |string                                                   |NULL   |
|state                       |string                                                   |NULL   |
|country                     |string                                                   |NULL   |
|registration_date           |string                                                   |NULL   |
|is_active                   |boolean                                                  |NULL   |
|                            |

In [7]:
!hadoop fs -ls -h /user/hive/warehouse/customers_300mb

Found 3 items
-rw-r--r--   2 root hadoop          0 2026-04-14 16:54 /user/hive/warehouse/customers_300mb/_SUCCESS
-rw-r--r--   2 root hadoop    115.0 M 2026-04-14 16:54 /user/hive/warehouse/customers_300mb/part-00000-fdf09854-25ac-42a7-86b7-a6aa7fa24e6e-c000.csv
-rw-r--r--   2 root hadoop    111.0 M 2026-04-14 16:54 /user/hive/warehouse/customers_300mb/part-00001-fdf09854-25ac-42a7-86b7-a6aa7fa24e6e-c000.csv


In [10]:
spark.sql('select * from customers_300mb limit 5').show()

+------------+----------+------+-----------+-------+-----------------+---------+
|customers_id|      name|  city|      state|country|registration_date|is_active|
+------------+----------+------+-----------+-------+-----------------+---------+
|        NULL|      name|  city|      state|country|registration_date|     NULL|
|           0|Customer_0|  Pune|Maharashtra|  India|       2023-01-19|     true|
|           1|Customer_1|  Pune|West Bengal|  India|       2023-08-10|     true|
|           2|Customer_2| Delhi|Maharashtra|  India|       2023-08-05|     true|
|           3|Customer_3|Mumbai|  Telangana|  India|       2023-06-04|     true|
+------------+----------+------+-----------+-------+-----------------+---------+



In [12]:
spark.sql('describe extended customers_300mb').show(truncate =False)

+----------------------------+---------------------------------------------------------+-------+
|col_name                    |data_type                                                |comment|
+----------------------------+---------------------------------------------------------+-------+
|customers_id                |int                                                      |NULL   |
|name                        |string                                                   |NULL   |
|city                        |string                                                   |NULL   |
|state                       |string                                                   |NULL   |
|country                     |string                                                   |NULL   |
|registration_date           |string                                                   |NULL   |
|is_active                   |boolean                                                  |NULL   |
|                            |

#### See how caching happens

In [13]:
spark.sql('select count(*) from customers_300mb').show() ## this is slow because each partition is giving answer and we are combining all the answers into one ouput

+--------+
|count(1)|
+--------+
| 3659892|
+--------+



#### 1. In the above query, each partition is giving it's own answer and we are combining all the answers into one ouput by using count, so it's taking time

#### Caching the table

In [14]:
spark.sql('cache table default.customers_300mb') #Eager Caching

DataFrame[]

In [15]:
spark.sql('select count(*) from default.customers_300mb').show() ## this is very fast because it's in the cache

+--------+
|count(1)|
+--------+
| 3659892|
+--------+



In [16]:
spark.sql('select (*) from default.customers_300mb limit 5').show() ## directly from cache so fast

+------------+----------+------+-----------+-------+-----------------+---------+
|customers_id|      name|  city|      state|country|registration_date|is_active|
+------------+----------+------+-----------+-------+-----------------+---------+
|        NULL|      name|  city|      state|country|registration_date|     NULL|
|           0|Customer_0|  Pune|Maharashtra|  India|       2023-01-19|     true|
|           1|Customer_1|  Pune|West Bengal|  India|       2023-08-10|     true|
|           2|Customer_2| Delhi|Maharashtra|  India|       2023-08-05|     true|
|           3|Customer_3|Mumbai|  Telangana|  India|       2023-06-04|     true|
+------------+----------+------+-----------+-------+-----------------+---------+



In [17]:
spark.sql('select city,count(*) from customers_300mb group by city').show() 

+---------+--------+
|     city|count(1)|
+---------+--------+
|    Delhi|  458095|
|  Kolkata|  457001|
|Hyderabad|  458259|
|     city|       1|
|Bangalore|  457291|
|Ahmedabad|  457709|
|  Chennai|  456714|
|   Mumbai|  457906|
|     Pune|  456916|
+---------+--------+



#### To uncache the table

In [18]:
spark.sql('uncache table customers_300mb')

DataFrame[]

### To cache the table lazily like dataframe/RDD

In [19]:
spark.sql('cache lazy table customers_300mb')

DataFrame[]

In [20]:
### This will be very slow
spark.sql('select count(*) from customers_300mb').show()

+--------+
|count(1)|
+--------+
| 3659892|
+--------+



#### The above query is very slow, because it should get the results and it should also cache the data because we did cache lazy in the above "cache lazy table"

In [21]:
spark.sql('show tables').show()

+---------+--------------------+-----------+
|namespace|           tableName|isTemporary|
+---------+--------------------+-----------+
|  default|           customers|      false|
|  default|     customers_300mb|      false|
|  default|customers_persistent|      false|
|  default|  external_customers|      false|
|  default|   managed_customers|      false|
+---------+--------------------+-----------+



## External Table

In [22]:
spark.sql('select * from external_customers').show()

+-----------+-----------+---------+-----------+-------+-----------------+---------+
|        _c0|        _c1|      _c2|        _c3|    _c4|              _c5|      _c6|
+-----------+-----------+---------+-----------+-------+-----------------+---------+
|customer_id|       name|     city|      state|country|registration_date|is_active|
|          0| Customer_0|     Pune|Maharashtra|  India|       2023-06-29|    False|
|          1| Customer_1|Bangalore| Tamil Nadu|  India|       2023-12-07|     True|
|          2| Customer_2|Hyderabad|    Gujarat|  India|       2023-10-27|     True|
|          3| Customer_3|Bangalore|  Karnataka|  India|       2023-10-17|    False|
|          4| Customer_4|Ahmedabad|  Karnataka|  India|       2023-03-14|    False|
|          5| Customer_5|Hyderabad|  Karnataka|  India|       2023-07-28|    False|
|          6| Customer_6|     Pune|      Delhi|  India|       2023-08-29|    False|
|          7| Customer_7|Ahmedabad|West Bengal|  India|       2023-12-28|   

In [24]:
spark.sql('describe extended external_customers').show(truncate =False)

+----------------------------+--------------------------------------------------+-------+
|col_name                    |data_type                                         |comment|
+----------------------------+--------------------------------------------------+-------+
|_c0                         |string                                            |NULL   |
|_c1                         |string                                            |NULL   |
|_c2                         |string                                            |NULL   |
|_c3                         |string                                            |NULL   |
|_c4                         |string                                            |NULL   |
|_c5                         |string                                            |NULL   |
|_c6                         |string                                            |NULL   |
|                            |                                                  |       |
|# Detaile

In [25]:
spark.sql('cache table external_customers')

DataFrame[]

#### To clear the cache everywhere

In [26]:
spark.catalog.clearCache() ## To check go to storage in Spark UI

In [27]:
spark.stop()